In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rai_compliance import GovernanceFramework



In [4]:
from rai_compliance import metrics



In [5]:
print("1. Loading the governance framework...")
framework = GovernanceFramework.from_yaml("example_manifest.yaml")
print("Framework loaded successfully!")

1. Loading the governance framework...
Framework loaded successfully!


In [6]:
# Step 2: Create or load a sample dataset
def create_sample_data(rows=100, cols=5, missing_ratio=0.1, outlier_ratio=0.05):
    """Create a sample DataFrame with missing values and outliers."""
    # Create a dictionary to hold the data
    data_dict = {}

    # Create columns with numeric data
    for i in range(cols):
        # Generate random data
        values = np.random.normal(0, 1, rows)

        # Add outliers
        outlier_indices = np.random.choice(rows, int(rows * outlier_ratio), replace=False)
        values[outlier_indices] = np.random.normal(10, 2, len(outlier_indices))

        # Add missing values
        missing_indices = np.random.choice(rows, int(rows * missing_ratio), replace=False)
        values[missing_indices] = np.nan

        # Add to dictionary
        data_dict[f"col_{i}"] = values

    # Add a target column
    data_dict["target"] = np.random.choice([0, 1], rows)

    # Create a pandas DataFrame
    df = pd.DataFrame(data_dict)

    return df

print("2. Creating sample dataset...")
data = create_sample_data(missing_ratio=0.15, outlier_ratio=0.1)
print("Dataset created with shape:", data.shape)
print(data.head())

2. Creating sample dataset...
Dataset created with shape: (100, 6)
       col_0     col_1      col_2      col_3     col_4  target
0  -1.513462 -0.454166  11.726988  -1.637404  0.914868       1
1   0.670611  0.331782        NaN   8.086975  0.402585       1
2  -0.934321 -0.757047   0.322917   0.076510 -1.080977       0
3  13.789130  0.226342   0.017395  -1.275013  0.104042       0
4  -1.292285 -1.457317   0.434704  14.225117       NaN       0


In [7]:
# Step 3: Validate the manifest against its schema
print("\n3. Validating manifest...")
from rai_compliance.validate_manifest import validate_manifest
validate_manifest("example_manifest.yaml")


3. Validating manifest...
✅ Manifest is valid against the schema!

=== Metrics Validation ===
Total metrics referenced: 7
Metrics available in registry: 7
Metrics available via connectors: 0
Missing metrics: 0

ℹ️  Available metric implementations:
  - accuracy
  - auc_roc
  - demographic_parity
  - equal_opportunity
  - f1_score
  - missing_values
  - outlier_detection
  - pii_detection
  - schema_validation

✅ All metrics have implementations!


True

In [8]:
# Step 4: Test data quality metrics for data ingestion stage
print("\n4. Evaluating compliance for data ingestion stage...")
report = framework.evaluate_compliance(
    model=None,  # No model needed for data quality checks
    dataset=data,
    stage="data_ingestion"
)


4. Evaluating compliance for data ingestion stage...


In [9]:
# Step 5: Analyze the report
print("\nCompliance Report Summary:")
summary = report.summary()
print(f"Overall Status: {'PASSED' if summary['passed'] else 'FAILED'}")
print(f"Rules: {summary['passed_rules']}/{summary['total_rules']} passed")
print(f"Tests: {summary['passed_tests']}/{summary['total_tests']} passed, {summary['failed_tests']} failed, {summary['skipped_tests']} skipped")

# Step 6: View detailed results
print("\nDetailed Results:")
for rule_id, rule_result in report.results.items():
    print(f"\nRule: {rule_result['name']} ({'PASSED' if rule_result['passed'] else 'FAILED'})")
    print(f"Description: {rule_result['description']}")

    for test_id, test_result in rule_result['tests'].items():
        status = "SKIPPED" if test_result.get('skipped', False) else "PASSED" if test_result['passed'] else "FAILED"
        print(f"\n  Test: {test_id} ({status})")
        print(f"  Value: {test_result['value']}")
        print(f"  Threshold: {test_result['threshold']}")

        if 'details' in test_result and test_result['details']:
            print("  Details:")
            for key, value in test_result['details'].items():
                print(f"    {key}: {value}")


Compliance Report Summary:
Overall Status: FAILED
Rules: 0/1 passed
Tests: 1/2 passed, 1 failed, 0 skipped

Detailed Results:

Rule: Data Quality (FAILED)
Description: Ensure the ingested data meets quality standards.

  Test: missing_values (FAILED)
  Value: 0.125
  Threshold: 0.05
  Details:
    total_cells: 600
    missing_cells: 75
    columns_missing_ratio: {'col_0': np.float64(0.15), 'col_1': np.float64(0.15), 'col_2': np.float64(0.15), 'col_3': np.float64(0.15), 'col_4': np.float64(0.15), 'target': np.float64(0.0)}

  Test: outlier_detection (PASSED)
  Value: 0.08190476190476191
  Threshold: 0.1
  Details:
    total_cells: 525
    outlier_cells: 43
    columns: {'col_0': {'ratio': np.float64(0.1059), 'count': 9, 'lower_bound': -3.5504139309648357, 'upper_bound': 3.9027319610872357}, 'col_1': {'ratio': np.float64(0.0824), 'count': 7, 'lower_bound': -2.404774086685534, 'upper_bound': 2.494282260950818}, 'col_2': {'ratio': np.float64(0.1059), 'count': 9, 'lower_bound': -2.73551642

In [47]:
# Step 7: Export the report to different formats
print("\n5. Exporting report...")
import os
directory = "./example_reports"
if not os.path.exists(directory):
    os.mkdir(directory)
# Export to JSON
json_report = report.to_json(directory + "/" + "compliance_report.json")
print("Exported JSON report to compliance_report.json")

# Export to Markdown
md_report = report.to_markdown(directory + "/" + "compliance_report.md")
print("Exported Markdown report to compliance_report.md")

# Export to HTML
html_report = report.to_html(directory + "/" + "compliance_report.html")
print("Exported HTML report to compliance_report.html")

# Export to PDF
report.to_pdf(directory + "/" + "compliance_report.pdf")
print("Exported PDF report to compliance_report.pdf")




5. Exporting report...
Exported JSON report to compliance_report.json
Exported Markdown report to compliance_report.md
Exported HTML report to compliance_report.html
PDF report saved to: ./example_reports/compliance_report.pdf
Exported PDF report to compliance_report.pdf


In [11]:

# Step 8: List available metrics
print("\n6. Available metrics in the framework:")
registry = framework.metric_registry
metrics = registry.list_metrics()
for metric_id, info in metrics.items():
    data_types = info.get('supported_data_types', ['unknown'])
    data_types_str = ", ".join(data_types)
    print(f"  - {metric_id} (Supports: {data_types_str})")
    print(f"    {info['description'][:60]}..." if len(info['description']) > 60 else f"    {info['description']}")


6. Available metrics in the framework:
  - missing_values (Supports: tabular)
    Check for missing values in a dataset.
    
    This metric ...
  - outlier_detection (Supports: tabular)
    Check for outliers in a dataset.
    
    This metric calcul...
  - schema_validation (Supports: tabular)
    Validate the schema of a dataset.
    
    This metric check...
  - demographic_parity (Supports: tabular)
    Measure statistical parity across demographic groups.
    
 ...
  - equal_opportunity (Supports: tabular)
    Measure equal opportunity across demographic groups.
    
  ...
  - accuracy (Supports: tabular)
    Calculate the accuracy of a model.
    
    This metric calc...
  - f1_score (Supports: tabular)
    Calculate the F1 score of a model.
    
    This metric calc...
  - auc_roc (Supports: tabular)
    Calculate the Area Under the ROC Curve (AUC-ROC) of a model....
  - pii_detection (Supports: text)
    Check for personally identifiable information (PII) in text ...


In [12]:
from rai_compliance.metrics import register_metric, supports_data_types


In [ ]:
# Define a custom metric function
@register_metric("custom_distribution_check")
@supports_data_types("tabular")
def distribution_check(model, dataset, config=None):
    """
    Custom metric to check if a dataset's columns follow normal distribution.
    
    This metric uses Shapiro-Wilk test to check normality of specified columns.
    
    Args:
        model: Not used for this metric
        dataset: Pandas DataFrame or similar structure with data to check
        config: Configuration with optional 'columns' list to check specific columns
        
    Returns:
        Dictionary with evaluation results - the average normality score
    """
    from scipy import stats
    import pandas as pd

    # Configuration
    config = config or {}
    columns = config.get('columns', [])

    # Ensure dataset is a pandas DataFrame
    if not isinstance(dataset, pd.DataFrame):
        try:
            dataset = pd.DataFrame(dataset)
        except:
            return {
                "value": 0.0,
                "details": {"error": "Dataset must be convertible to a pandas DataFrame"},
                "passed": False
            }

    # Select columns to analyze
    if columns:
        # Check if columns exist
        missing_cols = [col for col in columns if col not in dataset.columns]
        if missing_cols:
            return {
                "value": 0.0,
                "details": {"error": f"Columns not found: {missing_cols}"},
                "passed": False
            }
        numeric_cols = [col for col in columns if pd.api.types.is_numeric_dtype(dataset[col])]
    else:
        # Use all numeric columns
        numeric_cols = dataset.select_dtypes(include=['number']).columns.tolist()

    if not numeric_cols:
        return {
            "value": 0.0,
            "details": {"warning": "No numeric columns found for analysis"},
            "passed": True
        }

    # Compute Shapiro-Wilk test for each column
    results = {}
    p_values = []

    for col in numeric_cols:
        # Skip columns with NaN values or with less than 3 records
        valid_data = dataset[col].dropna()
        if len(valid_data) < 3:
            results[col] = {"test_statistic": None, "p_value": None, "normal": False}
            continue

        # Get a sample for large datasets (Shapiro-Wilk limited to 5000 samples)
        if len(valid_data) > 5000:
            valid_data = valid_data.sample(5000, random_state=42)

        # Perform Shapiro-Wilk test
        statistic, p_value = stats.shapiro(valid_data)
        normal = p_value > 0.05  # p > 0.05 typically indicates normality

        results[col] = {
            "test_statistic": float(statistic),
            "p_value": float(p_value),
            "normal": normal
        }
        p_values.append(p_value)

    # Calculate the average p-value as the metric value
    # Higher values indicate more normal distributions
    avg_p_value = sum(p_values) / len(p_values) if p_values else 0

    # Calculate how many columns follow normal distribution
    normal_cols = sum(1 for result in results.values() if result.get("normal", False))
    ratio_normal = normal_cols / len(results) if results else 0

    return {
        "value": ratio_normal,  # Ratio of columns that follow normal distribution
        "details": {
            "columns_analyzed": len(results),
            "columns_normal": normal_cols,
            "column_results": results,
            "avg_p_value": avg_p_value
        },
        "passed": True  # The framework will update this based on criteria
    }



ValueError: Metric with ID 'custom_distribution_check' is already registered

In [33]:
# Step 2: Create a custom manifest file with our new metric
custom_manifest = """
manifest_version: "1.0"
ai_project: "Custom Metrics Demo"
description: "Demonstrating custom metrics in RAI Compliance framework"

metadata:
owner: "Demo Team"
contact: "demo@example.com"
created_at: "2025-03-25"
version: "1.0.0"
domain: ["education"]
model_type: "classification"
data_type: "tabular"

pipeline_stages:
  - name: "data_exploration"
    description: "Data exploration and statistical analysis"
    data_type: "tabular"
    rules: 
    - "data_distributions"

rules:
  - id: "data_distributions"
    name: "Data Distribution Analysis"
    description: "Check if data columns follow expected distributions."
    tests:
      - id: "custom_distribution_check"
        name: "Normal Distribution Check"
        description: "Check if numeric columns follow normal distribution."
        criteria:
          operator: ">="
          value: 0.3
        threshold: 0.3
        config:
          columns: []  # Empty to check all numeric columns
        skip: false
        skip_reason: ""
"""
# Save the custom manifest to a file
with open("custom_manifest.yaml", "w") as f:
    f.write(custom_manifest)

In [34]:
# Step 3: Create a dataset with different distributions
def create_mixed_distributions_dataset(rows=500):
    """Create a dataset with columns of different distributions."""
    data = {
        # Normal distribution
        "normal_data": np.random.normal(loc=0, scale=1, size=rows),

        # Uniform distribution
        "uniform_data": np.random.uniform(low=0, high=10, size=rows),

        # Exponential distribution
        "exponential_data": np.random.exponential(scale=2, size=rows),

        # Chi-squared distribution
        "chi_squared_data": np.random.chisquare(df=3, size=rows),

        # Bimodal distribution (mixture of two normals)
        "bimodal_data": np.concatenate([
            np.random.normal(loc=-3, scale=1, size=rows//2),
            np.random.normal(loc=3, scale=1, size=rows//2)
        ]),

        # Normal with outliers
        "normal_with_outliers": np.random.normal(loc=0, scale=1, size=rows)
    }

    # Add outliers to the last column
    outlier_indices = np.random.choice(rows, size=int(rows*0.05), replace=False)
    data["normal_with_outliers"][outlier_indices] = np.random.normal(loc=20, scale=5, size=len(outlier_indices))

    # Create a pandas DataFrame
    df = pd.DataFrame(data)

    # Add a target column
    df["target"] = np.random.choice([0, 1], size=rows)

    return df

# Create our dataset
data = create_mixed_distributions_dataset()

In [35]:
# Visualize the distributions
plt.figure(figsize=(15, 10))
for i, column in enumerate(data.columns[:-1], 1):  # Skip target column
    plt.subplot(2, 3, i)
    plt.hist(data[column], bins=30, alpha=0.7)
    plt.title(column)
    plt.tight_layout()
plt.savefig('distributions.png')
plt.close()


In [ ]:
# Load our custom manifest and evaluate
custom_framework = GovernanceFramework.from_yaml("custom_manifest.yaml")
print("Framework loaded with custom metric!")

# Run the compliance check
custom_report = custom_framework.evaluate_compliance(
    model=None,
    dataset=data,
    stage="data_exploration"
)

Framework loaded with custom metric!


In [37]:
# Step 5: Analyze the results
print("\nCompliance Report Summary:")
summary = custom_report.summary()
print(f"Overall Status: {'PASSED' if summary['passed'] else 'FAILED'}")
print(f"Rules: {summary['passed_rules']}/{summary['total_rules']} passed")
print(f"Tests: {summary['passed_tests']}/{summary['total_tests']} passed")

# Dive into the details
print("\nCustom Metric Results:")
for rule_id, rule_result in custom_report.results.items():
    for test_id, test_result in rule_result['tests'].items():
        if test_id == "custom_distribution_check":
            print(f"\nNormal Distribution Test Results:")
            print(f"Value (ratio of normal columns): {test_result['value']:.2f}")
            print(f"Threshold: {test_result['threshold']}")
            print(f"Status: {'PASSED' if test_result['passed'] else 'FAILED'}")

            # Show column-specific results
            print("\nColumn-specific results:")
            details = test_result['details']
            for col, col_result in details.get('column_results', {}).items():
                normal = col_result.get('normal', False)
                status = "✅ Normal" if normal else "❌ Not normal"
                p_value = col_result.get('p_value', 'N/A')
                if p_value != 'N/A':
                    p_value = f"{p_value:.4f}"
                print(f"  {col}: {status} (p-value: {p_value})")

# Export the report
custom_report.to_markdown("custom_metric_report.md")
print("\nReport exported to: custom_metric_report.md")




Compliance Report Summary:
Overall Status: FAILED
Rules: 0/1 passed
Tests: 0/1 passed

Custom Metric Results:

Normal Distribution Test Results:
Value (ratio of normal columns): 0.14
Threshold: 0.3
Status: FAILED

Column-specific results:
  normal_data: ✅ Normal (p-value: 0.8031)
  uniform_data: ❌ Not normal (p-value: 0.0000)
  exponential_data: ❌ Not normal (p-value: 0.0000)
  chi_squared_data: ❌ Not normal (p-value: 0.0000)
  bimodal_data: ❌ Not normal (p-value: 0.0000)
  normal_with_outliers: ❌ Not normal (p-value: 0.0000)
  target: ❌ Not normal (p-value: 0.0000)

Report exported to: custom_metric_report.md


In [38]:
"""
Connector for IBM AI Fairness 360 library.

This connector provides integration with the AIF360 library for fairness metrics.
"""
from typing import Any, Dict, Optional, Set, ClassVar, List, Tuple, Union
import logging
import importlib
import numpy as np

from rai_compliance.connectors import BaseConnector, register_connector
from rai_compliance.utils.data_helpers import to_numpy_if_possible

logger = logging.getLogger(__name__)

@register_connector
class AIF360Connector(BaseConnector):
    """
    Connector for IBM AI Fairness 360 library.
    
    This connector provides integration with the AIF360 library for fairness metrics.
    """
    id: ClassVar[str] = "aif360"
    name: ClassVar[str] = "AI Fairness 360"
    description: ClassVar[str] = "Connector for IBM AI Fairness 360 library"
    version: ClassVar[str] = "1.0.0"
    external_library: ClassVar[str] = "aif360"
    supported_metrics: ClassVar[Set[str]] = {
        "demographic_parity", 
        "equal_opportunity",
        "disparate_impact",
        "statistical_parity_difference",
        "average_odds_difference"
    }
    
    def __init__(self, config: Optional[Dict[str, Any]] = None):
        """Initialize the AIF360 connector."""
        super().__init__(config)
        self.aif360 = None
        self.fairness_metrics = {}
    
    def initialize(self) -> bool:
        """
        Initialize the connector and load the AIF360 library.
        
        Returns:
            True if initialization was successful, False otherwise
        """
        if self._is_initialized:
            return True
        
        try:
            self.aif360 = importlib.import_module("aif360")
            from aif360.metrics import BinaryLabelDatasetMetric
            from aif360.metrics import ClassificationMetric
            from aif360.datasets import BinaryLabelDataset
            
            self.fairness_metrics = {
                "demographic_parity": self._evaluate_demographic_parity,
                "equal_opportunity": self._evaluate_equal_opportunity,
                "disparate_impact": self._evaluate_disparate_impact,
                "statistical_parity_difference": self._evaluate_statistical_parity_difference,
                "average_odds_difference": self._evaluate_average_odds_difference
            }
            
            self._is_initialized = True
            return True
        except ImportError as e:
            logger.error(f"Failed to initialize AIF360 connector: {e}")
            return False
    
    def supports_metric(self, metric_id: str) -> bool:
        """
        Check if this connector supports a specific metric.
        
        Args:
            metric_id: The ID of the metric to check
            
        Returns:
            True if the connector supports the metric, False otherwise
        """
        return metric_id in self.supported_metrics
    
    def evaluate_metric(self, metric_id: str, model: Any, dataset: Any, config: Dict[str, Any]) -> Dict[str, Any]:
        """
        Evaluate a fairness metric using AIF360.
        
        Args:
            metric_id: The ID of the metric to evaluate
            model: The model to evaluate
            dataset: The dataset to use for evaluation
            config: Configuration for the metric
            
        Returns:
            A dictionary with the evaluation results
            
        Raises:
            ValueError: If the metric is not supported or inputs are invalid
        """
        if not self.is_available:
            if not self.initialize():
                raise ValueError(f"AIF360 library is not available")
        
        if not self.supports_metric(metric_id):
            raise ValueError(f"Metric '{metric_id}' is not supported by the AIF360 connector")
        
        # Validate config
        if "sensitive_attributes" not in config:
            raise ValueError("Config must include 'sensitive_attributes'")
        
        # Get metric function
        metric_func = self.fairness_metrics.get(metric_id)
        if not metric_func:
            raise ValueError(f"Implementation for metric '{metric_id}' not found")
        
        # Evaluate metric
        try:
            return metric_func(model, dataset, config)
        except Exception as e:
            logger.error(f"Error evaluating metric '{metric_id}': {e}")
            raise ValueError(f"Error evaluating metric '{metric_id}': {e}")
    
    def _prepare_data(self, dataset: Any, config: Dict[str, Any]) -> Tuple[Any, List[str], str]:
        """
        Prepare data for AIF360 metrics.
        
        Args:
            dataset: The dataset to prepare
            config: Configuration with sensitive_attributes and target_column
            
        Returns:
            Tuple of (prepared dataset, sensitive attribute names, target column name)
            
        Raises:
            ValueError: If inputs are invalid
        """
        # Convert to pandas if possible
        try:
            import pandas as pd
            if not isinstance(dataset, pd.DataFrame):
                raise ValueError("Dataset must be a pandas DataFrame")
        except ImportError:
            raise ValueError("pandas is required for AIF360 connector")
        
        # Get sensitive attributes and target column
        sensitive_attrs = config.get("sensitive_attributes", [])
        if not sensitive_attrs:
            raise ValueError("Config must include non-empty 'sensitive_attributes'")
        
        target_column = config.get("target_column")
        if not target_column:
            raise ValueError("Config must include 'target_column'")
        
        # Check if columns exist
        missing_cols = [col for col in sensitive_attrs + [target_column] if col not in dataset.columns]
        if missing_cols:
            raise ValueError(f"Columns not found in dataset: {missing_cols}")
        
        return dataset, sensitive_attrs, target_column
    
    def _create_aif360_dataset(self, dataset: Any, sensitive_attrs: List[str], target_column: str) -> Any:
        """
        Create an AIF360 BinaryLabelDataset from a pandas DataFrame.
        
        Args:
            dataset: pandas DataFrame
            sensitive_attrs: List of sensitive attribute column names
            target_column: Target column name
            
        Returns:
            AIF360 BinaryLabelDataset
        """
        from aif360.datasets import BinaryLabelDataset
        
        # Create feature list (all columns except sensitive and target)
        features = [col for col in dataset.columns if col not in sensitive_attrs + [target_column]]
        
        # Create dataset
        return BinaryLabelDataset(
            df=dataset,
            label_names=[target_column],
            protected_attribute_names=sensitive_attrs,
            favorable_label=1,
            unfavorable_label=0
        )
    
    def _evaluate_demographic_parity(self, model: Any, dataset: Any, config: Dict[str, Any]) -> Dict[str, Any]:
        """
        Evaluate demographic parity using AIF360.
        
        Args:
            model: The model to evaluate
            dataset: The dataset to use for evaluation
            config: Configuration for the metric
            
        Returns:
            A dictionary with the evaluation results
        """
        # Prepare data
        df, sensitive_attrs, target_column = self._prepare_data(dataset, config)
        
        # Get predictions from model
        try:
            # Extract features
            features = df.drop(columns=sensitive_attrs + [target_column])
            
            # Get predictions
            if callable(model):
                predictions = model(features)
            elif hasattr(model, 'predict'):
                predictions = model.predict(features)
            else:
                raise ValueError("Model must be callable or have predict method")
            
            # Create predicted dataset
            df_pred = df.copy()
            df_pred[target_column] = predictions
            
            # Create AIF360 datasets
            dataset_orig = self._create_aif360_dataset(df, sensitive_attrs, target_column)
            dataset_pred = self._create_aif360_dataset(df_pred, sensitive_attrs, target_column)
            
            # Compute metrics for each sensitive attribute
            results = {}
            max_disparity = 0.0
            
            for attr in sensitive_attrs:
                # Get privileged and unprivileged groups
                privileged_groups = [{attr: 1}]
                unprivileged_groups = [{attr: 0}]
                
                # Compute classification metrics
                metric = self.aif360.metrics.ClassificationMetric(
                    dataset_orig, dataset_pred,
                    unprivileged_groups=unprivileged_groups,
                    privileged_groups=privileged_groups
                )
                
                # Calculate statistical parity difference
                spd = metric.statistical_parity_difference()
                
                # Store results
                results[attr] = {
                    "disparity": float(abs(spd)),
                    "statistical_parity_difference": float(spd)
                }
                
                max_disparity = max(max_disparity, abs(spd))
            
            # Return results
            return {
                "value": float(max_disparity),
                "details": results,
                "passed": True  # The framework will update this based on criteria
            }
            
        except Exception as e:
            logger.error(f"Error in demographic parity calculation: {e}")
            raise ValueError(f"Error in demographic parity calculation: {e}")
    
    def _evaluate_equal_opportunity(self, model: Any, dataset: Any, config: Dict[str, Any]) -> Dict[str, Any]:
        """
        Evaluate equal opportunity using AIF360.
        
        Args:
            model: The model to evaluate
            dataset: The dataset to use for evaluation
            config: Configuration for the metric
            
        Returns:
            A dictionary with the evaluation results
        """
        # Prepare data
        df, sensitive_attrs, target_column = self._prepare_data(dataset, config)
        
        # Get predictions from model
        try:
            # Extract features
            features = df.drop(columns=sensitive_attrs + [target_column])
            
            # Get predictions
            if callable(model):
                predictions = model(features)
            elif hasattr(model, 'predict'):
                predictions = model.predict(features)
            else:
                raise ValueError("Model must be callable or have predict method")
            
            # Create predicted dataset
            df_pred = df.copy()
            df_pred[target_column] = predictions
            
            # Create AIF360 datasets
            dataset_orig = self._create_aif360_dataset(df, sensitive_attrs, target_column)
            dataset_pred = self._create_aif360_dataset(df_pred, sensitive_attrs, target_column)
            
            # Compute metrics for each sensitive attribute
            results = {}
            max_disparity = 0.0
            
            for attr in sensitive_attrs:
                # Get privileged and unprivileged groups
                privileged_groups = [{attr: 1}]
                unprivileged_groups = [{attr: 0}]
                
                # Compute classification metrics
                metric = self.aif360.metrics.ClassificationMetric(
                    dataset_orig, dataset_pred,
                    unprivileged_groups=unprivileged_groups,
                    privileged_groups=privileged_groups
                )
                
                # Calculate equal opportunity difference
                eod = metric.equal_opportunity_difference()
                
                # Store results
                results[attr] = {
                    "disparity": float(abs(eod)),
                    "equal_opportunity_difference": float(eod)
                }
                
                max_disparity = max(max_disparity, abs(eod))
            
            # Return results
            return {
                "value": float(max_disparity),
                "details": results,
                "passed": True  # The framework will update this based on criteria
            }
            
        except Exception as e:
            logger.error(f"Error in equal opportunity calculation: {e}")
            raise ValueError(f"Error in equal opportunity calculation: {e}")
    
    def _evaluate_disparate_impact(self, model: Any, dataset: Any, config: Dict[str, Any]) -> Dict[str, Any]:
        """
        Evaluate disparate impact using AIF360.
        
        Args:
            model: The model to evaluate
            dataset: The dataset to use for evaluation
            config: Configuration for the metric
            
        Returns:
            A dictionary with the evaluation results
        """
        # Similar implementation to other metrics, focusing on disparate impact
        # This is a placeholder - actual implementation would follow the same pattern
        return {
            "value": 0.0,
            "details": {"note": "Placeholder implementation"},
            "passed": True
        }
    
    def _evaluate_statistical_parity_difference(self, model: Any, dataset: Any, config: Dict[str, Any]) -> Dict[str, Any]:
        """
        Evaluate statistical parity difference using AIF360.
        
        Args:
            model: The model to evaluate
            dataset: The dataset to use for evaluation
            config: Configuration for the metric
            
        Returns:
            A dictionary with the evaluation results
        """
        # Similar implementation to demographic parity, but directly returns SPD
        # This is a placeholder - actual implementation would follow the same pattern
        return {
            "value": 0.0,
            "details": {"note": "Placeholder implementation"},
            "passed": True
        }
    
    def _evaluate_average_odds_difference(self, model: Any, dataset: Any, config: Dict[str, Any]) -> Dict[str, Any]:
        """
        Evaluate average odds difference using AIF360.
        
        Args:
            model: The model to evaluate
            dataset: The dataset to use for evaluation
            config: Configuration for the metric
            
        Returns:
            A dictionary with the evaluation results
        """
        # Similar implementation to other metrics, focusing on average odds difference
        # This is a placeholder - actual implementation would follow the same pattern
        return {
            "value": 0.0,
            "details": {"note": "Placeholder implementation"},
            "passed": True
        }

In [39]:
from rai_compliance.connectors import list_connectors

list_connectors()

{'aif360': {'name': 'AI Fairness 360',
  'description': 'Connector for IBM AI Fairness 360 library',
  'version': '1.0.0',
  'external_library': 'aif360',
  'supported_metrics': ['demographic_parity',
   'disparate_impact',
   'statistical_parity_difference',
   'average_odds_difference',
   'equal_opportunity']}}

In [40]:
from rai_compliance.metrics import MetricRegistry

registry = MetricRegistry()

In [43]:
print(registry.list_metrics().keys())

dict_keys(['missing_values', 'outlier_detection', 'schema_validation', 'demographic_parity', 'equal_opportunity', 'accuracy', 'f1_score', 'auc_roc', 'pii_detection', 'custom_distribution_check'])


In [44]:
from rai_compliance.connectors import get_connector

connector_class = get_connector("aif360")()

In [ ]:
result = connector_class.evaluate_metric("")